# Day 2 · Hands-on 3 — Model diagnostics
**About 30 min · work in pairs** (one drives, one reads, predicts and questions — swap halfway)


Every linear model makes assumptions: a **straight-line** relationship, **constant spread**, roughly **normal** residuals, and no single point **dominating** the fit. Diagnostic plots show when these fail. First we learn what failures look like on data where we planted them, then we check a real model.



> **Two AI roles — keep them apart**
> - 🧮 **Gemini in Colab = calculator.** Ask it to change, fix or explain *code*.
> - 🎓 **Claude / ChatGPT (other tab) = tutor you must check.** Ask it to *interpret*, then score its answer: **correct · complete · specific · actionable**.


**How this notebook works:** run cells from top to bottom. Each code cell has one line marked `# 🔧 CHANGE THIS` — the only thing you need to edit. Lines starting with 📋 go in the shared sheet. Lines starting with 🎯 appear only for the simulated dataset, where we know the true answer.

In [ ]:
# @title ▶️ Setup: run this cell first (no changes needed) { display-mode: "form" }
# Every analysis below uses ROLES (outcome, group, ...), never hard-coded column names.
# To add a dataset: write a loader function and add one entry to DATASETS.
%pip install -q lifelines
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
from lifelines.datasets import load_gbsg2, load_lung

def _gbsg2():
    d = load_gbsg2()
    for c in ["tgrade", "menostat", "horTh"]:
        d[c] = d[c].astype(str)
    d["nodes4"] = (d["pnodes"] >= 4).astype(int)
    return d

def _lung():
    d = load_lung()
    d.columns = [c.replace(".", "_") for c in d.columns]
    d = d.dropna(subset=["ph_ecog"])
    d["ecog"] = d["ph_ecog"].map({0: "0", 1: "1", 2: "2+", 3: "2+"})
    d["sex"] = d["sex"].map({1: "male", 2: "female"})
    d["lost_weight"] = np.where(d["wt_loss"].isna(), np.nan, (d["wt_loss"] > 0).astype(float))
    return d

def _simulated(seed=2026, n=300):
    rng = np.random.default_rng(seed)
    genotype = rng.choice(["AA", "AB", "BB"], size=n, p=[0.4, 0.4, 0.2])
    sex = rng.choice(["F", "M"], size=n)
    age = rng.uniform(30, 80, n) + np.where(genotype == "BB", 8, 0)          # BB carriers are older
    g_eff = pd.Series({"AA": 0, "AB": 3, "BB": 8})[genotype].to_numpy()
    slope = np.where(sex == "M", 0.7, 0.4)                                    # interaction: steeper in men
    biomarker = 50 + slope * (age - 50) + g_eff + rng.normal(0, 6, n)
    logit = -1 + 0.05 * (age - 50) + pd.Series({"AA": 0, "AB": 0.4, "BB": 1.0})[genotype].to_numpy()
    disease = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    mu = np.exp(0.5 + 0.02 * (age - 50) + pd.Series({"AA": 0, "AB": 0.3, "BB": 0.7})[genotype].to_numpy())
    alpha = 0.8
    lesions = rng.negative_binomial(1 / alpha, 1 / (1 + alpha * mu))
    return pd.DataFrame(dict(biomarker=biomarker.round(1), genotype=genotype, sex=sex, age=age.round(1),
                             disease=disease, lesions=lesions))

def _my_data():
    # Upload a CSV to Colab (folder icon on the left) and edit the roles in DATASETS["my_data"]
    return pd.read_csv("my_data.csv")

DATASETS = {
    "gbsg2": dict(load=_gbsg2,
        about="GBSG2 breast cancer trial: 686 patients",
        outcome="tsize", outcome_label="Tumour size (mm)",
        group="tgrade", group_label="Tumour grade",
        x="pnodes", x_label="Positive lymph nodes",
        group2="menostat", group2_label="Menopausal status",
        binary="nodes4", binary_label="4 or more positive lymph nodes",
        binary_x="tsize", binary_group="tgrade",
        count="pnodes", count_label="Positive lymph nodes", count_x="tsize", count_group="tgrade"),
    "lung": dict(load=_lung,
        about="NCCTG advanced lung cancer study: 227 patients",
        outcome="pat_karno", outcome_label="Patient-rated Karnofsky score",
        group="ecog", group_label="ECOG performance status",
        x="age", x_label="Age (years)",
        group2="sex", group2_label="Sex",
        binary="lost_weight", binary_label="Lost weight in last 6 months",
        binary_x="age", binary_group="ecog",
        count=None),
    "simulated": dict(load=_simulated,
        about="Simulated biomarker study: 300 people, true effects known",
        outcome="biomarker", outcome_label="Biomarker level",
        group="genotype", group_label="Genotype",
        x="age", x_label="Age (years)",
        group2="sex", group2_label="Sex",
        binary="disease", binary_label="Disease present",
        binary_x="age", binary_group="genotype",
        count="lesions", count_label="Number of lesions", count_x="age", count_group="genotype",
        truth={
            "group": "AB vs AA = +3, BB vs AA = +8 (adjusted for age). BB carriers are also ~8 years older.",
            "x": "age slope = 0.4 per year in women, 0.7 in men",
            "interaction": "difference in age slopes (M vs F) = +0.3 per year",
            "binary": "odds ratio per year of age = 1.05; AB vs AA = 1.49; BB vs AA = 2.72",
            "count": "rate ratio per year of age = 1.02; AB vs AA = 1.35; BB vs AA = 2.01; overdispersed (alpha = 0.8)",
        }),
    "my_data": dict(load=_my_data,
        about="Your own CSV: edit these roles to match your column names",
        outcome="my_outcome", outcome_label="My outcome",
        group="my_group", group_label="My group",
        x="my_numeric_predictor", x_label="My numeric predictor",
        group2="my_two_level_group", group2_label="My two-level group",
        binary="my_yes_no_outcome", binary_label="My yes/no outcome (coded 0/1)",
        binary_x="my_numeric_predictor", binary_group="my_group",
        count=None),
}

def load_dataset(name):
    R = dict(DATASETS[name])
    d = R.pop("load")()
    for role in ["group", "group2", "binary_group", "count_group"]:
        if R.get(role):
            d[R[role]] = d[R[role]].astype(str)
    R["levels"] = sorted(d[R["group"]].dropna().unique())
    R.setdefault("truth", {})
    print(f"✅ {R['about']}  ({len(d)} rows)")
    print(f"   outcome: {R['outcome']} ({R['outcome_label']})   group: {R['group']} {R['levels']}")
    print(f"   numeric predictor: {R['x']}   second group: {R['group2']}")
    return d, R

def cat(var, ref):
    """Categorical term with an explicit reference level."""
    return f"C({var}, Treatment(reference='{ref}'))"

def nice(term):
    """Turn statsmodels term names into readable labels."""
    m = re.fullmatch(r"C\((\w+), Treatment\(reference='([^']+)'\)\)\[T\.([^\]]+)\](.*)", term)
    if m:
        var, ref, lvl, rest = m.groups()
        label = f"{var}: {lvl} vs {ref}"
        return label + (f" × {rest.lstrip(':')}" if rest else "")
    m = re.fullmatch(r"(\w+):C\((\w+), Treatment\(reference='([^']+)'\)\)\[T\.([^\]]+)\]", term)
    if m:
        x, var, ref, lvl = m.groups()
        return f"{x} × {var}: {lvl} vs {ref}"
    return term

def tidy(fit, exponentiate=False):
    """Coefficient table: estimate, 95% CI, p-value."""
    ci = fit.conf_int()
    t = pd.DataFrame({"estimate": fit.params, "CI low": ci[0], "CI high": ci[1], "p-value": fit.pvalues})
    t = t.drop(index=[i for i in t.index if i == "alpha"], errors="ignore")
    if exponentiate:
        t[["estimate", "CI low", "CI high"]] = np.exp(t[["estimate", "CI low", "CI high"]])
    t.index = [nice(i) for i in t.index]
    return t.round({"estimate": 3, "CI low": 3, "CI high": 3}).assign(**{"p-value": t["p-value"].map(lambda p: f"{p:.2g}")})

def show_truth(R, key):
    if key in R["truth"]:
        print(f"🎯 Truth used to simulate: {R['truth'][key]}")

print("✅ Setup complete")

from statsmodels.nonparametric.smoothers_lowess import lowess

def diagnose(fit, title=""):
    """Four standard diagnostic plots."""
    infl = fit.get_influence()
    fitted, resid = fit.fittedvalues, fit.resid
    std_resid = infl.resid_studentized_internal
    fig, ax = plt.subplots(2, 2, figsize=(9, 6.5))
    ax[0, 0].scatter(fitted, resid, alpha=0.4, s=12)
    lw = lowess(resid, fitted)
    ax[0, 0].plot(lw[:, 0], lw[:, 1], color="red"); ax[0, 0].axhline(0, color="gray", ls="--")
    ax[0, 0].set(title="Residuals vs fitted  (look for: curve)", xlabel="Fitted", ylabel="Residual")
    stats.probplot(std_resid, plot=ax[0, 1]); ax[0, 1].set_title("Normal Q-Q  (look for: points off the line)")
    sl = np.sqrt(np.abs(std_resid))
    ax[1, 0].scatter(fitted, sl, alpha=0.4, s=12)
    lw = lowess(sl, fitted)
    ax[1, 0].plot(lw[:, 0], lw[:, 1], color="red")
    ax[1, 0].set(title="Scale-location  (look for: funnel)", xlabel="Fitted", ylabel="√|std. residual|")
    cooks = infl.cooks_distance[0]
    ax[1, 1].stem(np.arange(len(cooks)), cooks, markerfmt=" ")
    ax[1, 1].axhline(4 / len(cooks), color="red", ls="--")
    ax[1, 1].set(title="Cook's distance  (look for: spikes)", xlabel="Observation", ylabel="Cook's D")
    fig.suptitle(title); fig.tight_layout(); plt.show()

## A. A gallery of problems
### 🧭 Concept
Each option plants one problem in simulated data. Your facilitator assigns one to your pair; afterwards, look at all of them. For each, find the plot that gives it away.

In [ ]:
# 🔧 CHANGE THIS: "none", "curved", "funnel", "outlier" or "skewed"
problem = "none"

rng = np.random.default_rng(1)
x = rng.uniform(0, 10, 100)
noise = rng.normal(0, 2, 100)
if problem == "curved":
    y = 2 + 0.3 * x ** 2 + noise
elif problem == "funnel":
    y = 2 + 1.5 * x + noise * (0.1 + 0.4 * x)
elif problem == "skewed":
    y = 2 + 1.5 * x + (rng.exponential(3, 100) - 3)
else:
    y = 2 + 1.5 * x + noise
if problem == "outlier":
    x, y = np.append(x, 20), np.append(y, 5)          # one far-out point that pulls the line

sim = pd.DataFrame({"x": x, "y": y})
fit = smf.ols("y ~ x", data=sim).fit()
plt.figure(figsize=(4.5, 3)); plt.scatter(x, y, s=12, alpha=0.6)
xs = np.linspace(x.min(), x.max(), 50); plt.plot(xs, fit.params["Intercept"] + fit.params["x"] * xs, color="black")
plt.title(f"Data: problem = {problem}   (true slope 1.5, fitted {fit.params['x']:.2f})"); plt.show()
diagnose(fit, f"Diagnostics: problem = {problem}")

### ✅ Concept check
Match each problem to the plot that shows it most clearly.

<details><summary>Show answers</summary>

- **curved** → residuals vs fitted: a U-shaped red line. Fix: add a curve term, transform, or use a different model.
- **funnel** → scale-location: the red line rises. Fix: transform the outcome or use robust standard errors.
- **skewed** → Q-Q plot: points curve away at one end. With large samples this matters less than the other problems.
- **outlier** → Cook's distance: one tall spike, and the fitted slope moves away from 1.5. Fix: *check the data point first*, never delete it silently.
</details>

## Choose a dataset
Everything in this notebook is written in terms of **roles** (an outcome, a group, a numeric predictor), so the same cells work on any dataset. Start with the one your facilitator names, then try `simulated`: it was generated with known effects, so you can check whether the model recovers the truth.

In [ ]:
# 🔧 CHANGE THIS: which dataset to use (all cells below adapt automatically)
dataset = "gbsg2"  # @param ["gbsg2", "lung", "simulated", "my_data"]

df, R = load_dataset(dataset)
df.head()

## B. Check a real model
### 🧭 Concept
Now fit `outcome ~ x + group` on your dataset and read its diagnostics. If something looks wrong, try transforming the outcome. Skewed positive outcomes (sizes, concentrations, counts) often look better on the log scale.

In [ ]:
# 🔧 CHANGE THIS: outcome transformation — "none", "log" or "sqrt"
transform = "none"

y, x, g, ref = R["outcome"], R["x"], R["group"], R["levels"][0]
d = df.dropna(subset=[y, x, g])
if transform != "none" and (d[y] <= 0).any():
    print("⚠️ The outcome has zero or negative values; log/sqrt are not appropriate here.")
lhs = {"none": y, "log": f"np.log({y})", "sqrt": f"np.sqrt({y})"}[transform]
fit = smf.ols(f"{lhs} ~ {x} + {cat(g, ref)}", data=d).fit()
diagnose(fit, f"{lhs} ~ {x} + {g}")
display(tidy(fit))

### ✅ Concept check
After a log transform, how do you interpret a coefficient of 0.10?

<details><summary>Show answer</summary>

Each unit increase in the predictor is associated with the outcome being multiplied by exp(0.10) ≈ 1.105, i.e. about 10% higher. Effects become **ratios** instead of differences.
</details>

## C. ⭐ If you have time: how much do influential points matter?
Refit the model from section B without the observations with the largest Cook's distance. Do the conclusions change?

In [ ]:
# 🔧 CHANGE THIS: number of most influential observations to drop
n_remove = 5

cooks = fit.get_influence().cooks_distance[0]
keep = np.argsort(cooks)[: len(cooks) - n_remove]
refit = smf.ols(fit.model.formula, data=d.iloc[keep]).fit()
display(pd.DataFrame({"all data": tidy(fit)["estimate"], f"without top {n_remove}": tidy(refit)["estimate"]}))
print("Most influential rows:")
display(d.iloc[np.argsort(cooks)[-n_remove:]])

## 🤖 AI prompts
**🎓 Claude / ChatGPT (tutor) — a known failure point.** Paste:
> *My linear regression has 686 observations. A Shapiro–Wilk test on the residuals gives p = 0.001. Should I abandon linear regression and use a non-parametric test instead?*

Score with the rubric. A good answer says that with large samples formal normality tests flag trivial departures; that **plots** matter more than the test; that linearity and constant spread matter more than normality; and offers fixes (transformation, robust standard errors) rather than throwing away the model. Watch for answers that jump straight to "use Mann–Whitney/Kruskal–Wallis", which would lose your adjustment for other predictors.

**🧮 Gemini in Colab (calculator):** select the cell in section B and ask:
> *Refit this model with heteroscedasticity-robust standard errors (HC3) and show both tables.*

Check: did the estimates stay the same while the confidence intervals changed? They should.

## 💬 Discuss with your partner
- Which assumption violation would worry you most in your own data, and why?
- A colleague removes three "outliers" to get p < 0.05. What would you ask them?
- Why is "the Q-Q plot looks a bit off" usually less serious than "the residuals curve"?